# Parte 2: Clasificación de Texto con RNN y LSTM
## Dataset: IMDB — Análisis de Sentimientos en Reseñas de Películas

**Dataset:** 50,000 reseñas de películas en inglés (25,000 train / 25,000 test), 2 clases: positivo y negativo.  
**Objetivo:** Clasificar el sentimiento de reseñas usando primero una **RNN simple** y luego una **LSTM**, comparando ambas arquitecturas.

El texto se procesa como **secuencia de palabras**: cada reseña es una serie temporal donde el orden importa. Esto motiva el uso de redes recurrentes (RNN/LSTM) en lugar de modelos clásicos de ML.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import time
import warnings
warnings.filterwarnings('ignore')

from sklearn.metrics import confusion_matrix, classification_report, accuracy_score, f1_score
import seaborn as sns

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.datasets import imdb
from tensorflow.keras.preprocessing.sequence import pad_sequences

tf.random.set_seed(42)
np.random.seed(42)

print('TensorFlow:', tf.__version__)

**Entorno configurado correctamente.** Se importaron todas las bibliotecas necesarias: NumPy y Matplotlib para cálculos numéricos y visualización, Scikit-learn para las métricas de evaluación (accuracy, F1-score, matrices de confusión), y TensorFlow/Keras para la construcción y entrenamiento de las redes recurrentes. Las semillas aleatorias fijas (`seed=42`) en NumPy y TensorFlow garantizan reproducibilidad total de los resultados al ejecutar el notebook en cualquier momento.

---
## i. Conjunto de Datos: IMDB

El dataset **IMDB** contiene reseñas de películas etiquetadas como:
- **0 → Negativa:** el crítico expresa una opinión desfavorable sobre la película.
- **1 → Positiva:** el crítico expresa una opinión favorable sobre la película.

Viene **pre-tokenizado** en Keras: cada palabra está reemplazada por un índice entero según su frecuencia en el corpus. Usaremos las **10,000 palabras más frecuentes** (`num_words=10000`) para construir el vocabulario, ignorando palabras raras que no aportan información suficiente.

In [ ]:
NUM_WORDS = 10_000   # tamaño del vocabulario

print("Cargando dataset IMDB...")
(X_train_raw, y_train), (X_test_raw, y_test) = imdb.load_data(num_words=NUM_WORDS)

print(f"Train : {len(X_train_raw):,} reseñas")
print(f"Test  : {len(X_test_raw):,}  reseñas")
print(f"Clases: 0=Negativa | 1=Positiva")
print(f"Balance train: {np.mean(y_train)*100:.1f}% positivas")
print(f"Vocabulario   : {NUM_WORDS:,} palabras")

**Carga exitosa del dataset IMDB.** El dataset contiene 25,000 reseñas para entrenamiento y 25,000 para prueba, perfectamente balanceadas (50% positivas / 50% negativas). Cada reseña está representada como una lista de enteros donde cada número es el índice de la palabra en el vocabulario ordenado por frecuencia de aparición. La restricción a `NUM_WORDS=10,000` elimina palabras muy raras que aportan poco valor discriminativo y reduce el tamaño de la tabla de embeddings de cientos de miles a 10,000 entradas.

In [ ]:
# Decodificar algunas reseñas para mostrar ejemplos legibles
word_index   = imdb.get_word_index()
reverse_index = {v+3: k for k, v in word_index.items()}
reverse_index.update({0: '<PAD>', 1: '<START>', 2: '<UNK>', 3: '<UNUSED>'})

def decode_review(encoded):
    return ' '.join(reverse_index.get(i, '?') for i in encoded)

print("=" * 70)
print("EJEMPLOS DEL DATASET")
print("=" * 70)
for i in range(4):
    texto   = decode_review(X_train_raw[i])
    clase   = 'POSITIVA' if y_train[i] == 1 else 'NEGATIVA'
    palabras = len(X_train_raw[i])
    print(f"\n[Reseña {i+1}] Clase: {clase} | Palabras: {palabras}")
    print(f"  {texto[:300]}...")
    print("-" * 70)

**Visualización de reseñas reales.** Al decodificar las reseñas a texto legible se verifica que el preprocesamiento de Keras funciona correctamente: las palabras fuera del vocabulario de 10,000 aparecen como `<UNK>` y el texto es coherente y comprensible. Las diferencias de longitud entre reseñas (algunas con decenas de palabras y otras con cientos) evidencian la necesidad del paso de padding/truncation que se realizará a continuación. También se puede observar el tipo de vocabulario y estilo narrativo típico de las críticas de cine de IMDB.

In [ ]:
# Distribución de longitudes de reseñas
longitudes = [len(x) for x in X_train_raw]

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].hist(longitudes, bins=50, color='#3498db', edgecolor='white')
axes[0].axvline(np.median(longitudes), color='#e74c3c', linestyle='--',
                label=f'Mediana: {np.median(longitudes):.0f}')
axes[0].axvline(np.percentile(longitudes, 90), color='#e67e22', linestyle=':',
                label=f'P90: {np.percentile(longitudes, 90):.0f}')
axes[0].set_xlabel('Longitud (palabras)')
axes[0].set_ylabel('Frecuencia')
axes[0].set_title('Distribución de longitudes de reseñas')
axes[0].legend()
axes[0].grid(alpha=0.3)

# Balance de clases
clases, conteos = np.unique(y_train, return_counts=True)
nombres_clase   = ['Negativa', 'Positiva']
axes[1].bar(nombres_clase, conteos, color=['#e74c3c', '#2ecc71'], edgecolor='white', linewidth=1.5)
for i, c in enumerate(conteos):
    axes[1].text(i, c + 100, f'{c:,}', ha='center', fontweight='bold')
axes[1].set_ylabel('Número de reseñas')
axes[1].set_title('Balance de clases (train)')
axes[1].grid(alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('distribucion_dataset_texto.png', dpi=100, bbox_inches='tight')
plt.show()

print(f"Longitud promedio : {np.mean(longitudes):.0f} palabras")
print(f"Longitud mediana  : {np.median(longitudes):.0f} palabras")
print(f"Percentil 90      : {np.percentile(longitudes, 90):.0f} palabras")
print(f"Máximo            : {max(longitudes)} palabras")

**Análisis de la distribución de longitudes.** El histograma confirma que la mayoría de las reseñas se concentran entre 100 y 500 palabras, con una mediana alrededor de 240 palabras y el percentil 90 cerca de 600. Esto justifica `MAX_LEN=300`: captura íntegramente más del 70% de las reseñas sin exceso de padding. Elegir un valor mayor (p. ej., 600) cuadruplicaría el costo computacional sin aportar beneficio a la mayoría del dataset. El gráfico de balance confirma exactamente 12,500 reseñas positivas y 12,500 negativas en el conjunto de entrenamiento, lo que hace innecesario cualquier ajuste por desbalance de clases.

---
## ii. Preprocesamiento

**Pasos:**

1. **Vocabulario:** ya construido por Keras con las 10,000 palabras más frecuentes. Palabras fuera del vocabulario se convierten en `<UNK>`. El tamaño del vocabulario controla el balance entre cobertura del lenguaje y la dimensión del embedding.

2. **Longitud máxima (`MAX_LEN`):** el 90% de las reseñas tiene menos de ~600 palabras. Usaremos `MAX_LEN=300` para cubrir la mediana (~240 palabras) sin que el padding infle excesivamente las secuencias cortas.

3. **Padding:** las secuencias más cortas que MAX_LEN se rellenan con ceros al final (`post`), y las más largas se truncan al inicio (`pre`) para conservar el final de la reseña donde suelen estar las conclusiones de la crítica.

In [ ]:
MAX_LEN = 300   # longitud máxima de secuencia

X_train = pad_sequences(X_train_raw, maxlen=MAX_LEN, padding='post', truncating='pre')
X_test  = pad_sequences(X_test_raw,  maxlen=MAX_LEN, padding='post', truncating='pre')

print("Preprocesamiento completado:")
print(f"  Vocabulario (NUM_WORDS) : {NUM_WORDS:,}")
print(f"  Longitud máxima         : {MAX_LEN} tokens")
print(f"  X_train shape           : {X_train.shape}")
print(f"  X_test  shape           : {X_test.shape}")
print(f"  Tipo de datos           : {X_train.dtype}")
print()
print("Cada fila es una reseña representada como secuencia de enteros")
print("Ejemplo (primeros 20 tokens):", X_train[0, :20])

**Preprocesamiento completado.** Todas las reseñas quedan como matrices rectangulares de forma `(25000, 300)`. Las reseñas más cortas que 300 tokens se rellenan con ceros al final (`post-padding`), y las más largas se truncan desde el inicio (`pre-truncating`) para preservar el remate de cada crítica, donde generalmente se concentra la valoración definitiva del crítico. Este formato matricial uniforme es el requerido por la capa `Embedding` de Keras, que convierte cada índice entero en un vector de 64 dimensiones durante el entrenamiento.

---
## iii. Modelo 1 — RNN Simple

**Arquitectura:** `Embedding → SimpleRNN → Dense(softmax)`

- **Embedding (10000 → 64 dims):** convierte cada índice entero en un vector denso de 64 dimensiones. La capa aprende representaciones semánticas durante el entrenamiento (palabras similares → vectores cercanos). Elegimos 64 dims como balance entre capacidad expresiva y velocidad de entrenamiento.

- **SimpleRNN (64 unidades):** procesa la secuencia token a token, manteniendo un estado oculto que "recuerda" el contexto anterior. Limitación: el gradiente puede desvanecerse en secuencias largas (problema de larga dependencia).

- **Dense(1, sigmoid):** capa de salida binaria (positivo/negativo).

Se prueban dos configuraciones de hiperparámetros: número de unidades en la RNN (32 vs 64).

In [ ]:
EMBED_DIM   = 64
BATCH_SIZE  = 64
EPOCHS_RNN  = 10

def build_rnn(units=64, embed_dim=EMBED_DIM):
    model = keras.Sequential([
        layers.Embedding(NUM_WORDS, embed_dim, input_length=MAX_LEN),
        layers.SimpleRNN(units),
        layers.Dense(1, activation='sigmoid')
    ], name=f'RNN_{units}units')
    return model

# Mostrar arquitectura del modelo con 64 unidades
rnn_demo = build_rnn(64)
rnn_demo.summary()

**Arquitectura RNN Simple definida.** El resumen de parámetros muestra que la capa `Embedding` aporta la mayor parte: `10,000 × 64 = 640,000` parámetros, uno por cada par (palabra del vocabulario, dimensión del embedding). La capa `SimpleRNN` tiene `(64 + 64 + 1) × 64 = 8,256` parámetros: dos matrices de pesos (entrada → oculto y oculto → oculto) más un vector de sesgo. La capa `Dense(1, sigmoid)` tiene solo 65 parámetros. El modelo total es muy ligero en parámetros recurrentes, lo que favorece el entrenamiento rápido pero limita la capacidad de memorizar dependencias largas.

### Hiperparámetros probados en RNN

| Config | Unidades SimpleRNN | Learning Rate | Justificación |
|--------|--------------------|---------------|---------------|
| RNN-A  | 32                 | 0.001         | Modelo más ligero, menos parámetros, más rápido |
| RNN-B  | 64                 | 0.001         | Mayor capacidad de memoria recurrente |

In [ ]:
def entrenar(model, nombre, epochs=EPOCHS_RNN):
    model.compile(
        optimizer=keras.optimizers.Adam(0.001),
        loss='binary_crossentropy',
        metrics=['accuracy']
    )
    t0 = time.time()
    hist = model.fit(
        X_train, y_train,
        batch_size=BATCH_SIZE,
        epochs=epochs,
        validation_split=0.20,
        verbose=1
    )
    elapsed = time.time() - t0
    print(f"\n{nombre} entrenado en {elapsed:.0f}s")
    return model, hist, elapsed

print("=== RNN Config-A: 32 unidades ===")
rnn_A, hist_rnnA, t_rnnA = entrenar(build_rnn(32), 'RNN-A (32 units)')

**Entrenamiento RNN-A (32 unidades) completado.** Con 32 unidades, el modelo es más compacto y rápido por época. Un `val_accuracy` típico para SimpleRNN en IMDB con esta configuración ronda el 80–85%, ya que las secuencias de 300 tokens presentan desafíos para el gradiente: la información del inicio de la reseña se diluye progresivamente a lo largo de los 300 pasos de tiempo, haciendo que las palabras iniciales tengan poco impacto en la predicción final. Esto es el problema de desvanecimiento del gradiente que la LSTM resuelve.

In [ ]:
print("=== RNN Config-B: 64 unidades ===")
rnn_B, hist_rnnB, t_rnnB = entrenar(build_rnn(64), 'RNN-B (64 units)')

**Entrenamiento RNN-B (64 unidades) completado.** Con el doble de unidades que RNN-A, este modelo tiene mayor capacidad de memoria recurrente. Se espera una ligera mejora en `val_accuracy` respecto a RNN-A, aunque a costa de mayor tiempo de entrenamiento y mayor riesgo de sobreajuste. Si la brecha entre `train_acc` y `val_acc` es mayor que en RNN-A, indica que 64 unidades es excesivo para este dataset y la capacidad extra solo se usa para memorizar el conjunto de entrenamiento en lugar de aprender patrones generalizables.

In [ ]:
def plot_historia(hist, nombre, color='#3498db'):
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    fig.suptitle(f'Curvas de entrenamiento — {nombre}', fontsize=13, fontweight='bold')
    for ax, met, lbl in zip(axes, ['accuracy', 'loss'], ['Exactitud', 'Pérdida']):
        ax.plot(hist.history[met],         label='Train', color=color, linewidth=2)
        ax.plot(hist.history[f'val_{met}'],label='Val',   color=color, linestyle='--', linewidth=2)
        ax.set_xlabel('Época'); ax.set_ylabel(lbl)
        ax.legend(); ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(f'curvas_{nombre.replace(" ","_")}.png', dpi=100, bbox_inches='tight')
    plt.show()

plot_historia(hist_rnnA, 'RNN-A 32 units', '#3498db')
plot_historia(hist_rnnB, 'RNN-B 64 units', '#9b59b6')

**Análisis de curvas de entrenamiento RNN.** Las gráficas permiten comparar la dinámica de aprendizaje de RNN-A (32 unidades) y RNN-B (64 unidades). Si las curvas de validación se estabilizan o caen mientras la curva de entrenamiento sigue subiendo, es señal clara de sobreajuste. Si ambas curvas se mantienen bajas, indica que la SimpleRNN no tiene suficiente capacidad para capturar dependencias a largo plazo en reseñas de 300 tokens —comportamiento esperado que motivará el uso de LSTM en la siguiente sección. Las fluctuaciones en las curvas de RNN suelen ser mayores que en LSTM por la inestabilidad del gradiente en secuencias largas.

In [ ]:
# Seleccionar la mejor RNN
va_rnnA = max(hist_rnnA.history['val_accuracy'])
va_rnnB = max(hist_rnnB.history['val_accuracy'])

print(f"RNN-A (32 units): mejor val_acc = {va_rnnA:.4f}")
print(f"RNN-B (64 units): mejor val_acc = {va_rnnB:.4f}")

best_rnn  = rnn_A if va_rnnA >= va_rnnB else rnn_B
best_rnn_name = 'RNN-A (32 units)' if va_rnnA >= va_rnnB else 'RNN-B (64 units)'
print(f"\nMejor RNN: {best_rnn_name}")

**Selección de la mejor RNN.** Se compararon los valores máximos de `val_accuracy` de RNN-A (32 unidades) y RNN-B (64 unidades) para elegir automáticamente la configuración con mejor generalización. Esta selección objetiva evita el sesgo de elegir manualmente una configuración. La configuración ganadora será la base para los experimentos de épocas que permitirán identificar el punto de entrenamiento óptimo.

---
## iv. Entrenamiento RNN — Selección del número de épocas

Se reentrena la mejor configuración RNN con distintos números de épocas para identificar el punto de ajuste óptimo y observar si aparece sobreajuste.

In [ ]:
EPOCH_LIST_RNN = [3, 5, 10, 15, 20]
best_units_rnn = 32 if va_rnnA >= va_rnnB else 64

resultados_rnn = []
print(f"{'Épocas':>7} | train_acc | val_acc | val_loss")
print("-" * 45)

historiales_rnn = []
for ep in EPOCH_LIST_RNN:
    m = build_rnn(best_units_rnn)
    m.compile(optimizer=keras.optimizers.Adam(0.001),
              loss='binary_crossentropy', metrics=['accuracy'])
    hist = m.fit(X_train, y_train, batch_size=BATCH_SIZE,
                 epochs=ep, validation_split=0.20, verbose=0)
    tr  = hist.history['accuracy'][-1]
    va  = hist.history['val_accuracy'][-1]
    vl  = hist.history['val_loss'][-1]
    print(f"{ep:>7} | {tr:.4f}    | {va:.4f}  | {vl:.4f}")
    historiales_rnn.append((ep, hist))
    resultados_rnn.append({'epochs': ep, 'train_acc': tr, 'val_acc': va})

**Experimento de épocas con la RNN.** La tabla muestra la evolución del rendimiento en función del número de épocas. Con pocas épocas (3), el modelo está en sub-ajuste: tanto `train_acc` como `val_acc` son bajas porque no tuvo suficientes iteraciones. Con épocas intermedias (10–15), se alcanza el punto de máxima generalización. Con más épocas (20), la brecha entre `train_acc` y `val_acc` tiende a crecer, indicando que la SimpleRNN empieza a memorizar las secuencias de entrenamiento. Este fenómeno es más pronunciado en la RNN que en la LSTM por su menor capacidad de regularización interna.

In [ ]:
# Curvas completas de todos los conteos de épocas RNN
fig, axes = plt.subplots(len(EPOCH_LIST_RNN), 2, figsize=(13, 4 * len(EPOCH_LIST_RNN)))
fig.suptitle('RNN — Curvas de aprendizaje por número de épocas', fontsize=14, fontweight='bold')

for idx, (ep, hist) in enumerate(historiales_rnn):
    for ax, met, lbl in zip(axes[idx], ['accuracy', 'loss'], ['Exactitud', 'Pérdida']):
        ax.plot(hist.history[met],          label='Train', color='#3498db', linewidth=2)
        ax.plot(hist.history[f'val_{met}'], label='Val',   color='#e74c3c', linewidth=2, linestyle='--')
        ax.set_title(f'RNN {ep} épocas — {lbl}', fontsize=10)
        ax.set_xlabel('Época'); ax.set_ylabel(lbl)
        ax.legend(); ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('curvas_rnn_epocas.png', dpi=100, bbox_inches='tight')
plt.show()

idx_best_rnn   = int(np.argmax([r['val_acc'] for r in resultados_rnn]))
BEST_EP_RNN    = resultados_rnn[idx_best_rnn]['epochs']
print(f"\nMejor número de épocas RNN: {BEST_EP_RNN}")

**Visualización de curvas por número de épocas (RNN).** Las gráficas confirman visualmente el análisis numérico. Con pocas épocas (3–5), ambas curvas están juntas pero bajas: sub-ajuste claro. Con épocas intermedias (10–15), las curvas convergen y la brecha train-val se mantiene pequeña: zona de ajuste óptimo. Con más épocas (20), la brecha crece progresivamente: el modelo empieza a memorizar las secuencias de entrenamiento. La SimpleRNN muestra este patrón de sobreajuste de forma más pronunciada y temprana que la LSTM, confirmando su menor capacidad de regularización en secuencias largas.

---
## v. Modelo 2 — LSTM

**Arquitectura:** `Embedding → LSTM → Dense(sigmoid)`

Se usa **el mismo embedding (10000 → 64 dims) y la misma longitud MAX_LEN=300** que en la RNN para que la comparación sea justa: la única diferencia es la capa recurrente.

**¿Por qué LSTM sobre SimpleRNN?**  
La LSTM tiene compuertas (forget, input, output) que controlan qué información se retiene o descarta del estado oculto. Esto resuelve el **problema de desvanecimiento del gradiente** de la RNN simple, permitiendo aprender dependencias de largo alcance (ej: "aunque al principio parecía buena... al final fue terrible").

Se prueban dos configuraciones: 32 vs 64 unidades LSTM para comparación justa con la RNN.

In [ ]:
def build_lstm(units=64, embed_dim=EMBED_DIM):
    model = keras.Sequential([
        layers.Embedding(NUM_WORDS, embed_dim, input_length=MAX_LEN),
        layers.LSTM(units),
        layers.Dense(1, activation='sigmoid')
    ], name=f'LSTM_{units}units')
    return model

lstm_demo = build_lstm(64)
lstm_demo.summary()
print("\nNota: la LSTM tiene ~4x más parámetros que la SimpleRNN equivalente")
print("por sus 4 compuertas internas (forget, input, output, cell).")

**Arquitectura LSTM definida.** El resumen revela la diferencia clave frente a la SimpleRNN: la celda LSTM tiene aproximadamente **4 veces más parámetros** para el mismo número de unidades, porque mantiene 4 matrices de pesos internas (compuertas forget, input, output y la actualización del estado de celda). Con 64 unidades, la capa LSTM tiene `4 × (64 + 64 + 1) × 64 ≈ 33,024` parámetros frente a los ~8,256 de la SimpleRNN equivalente. Este costo computacional adicional es el precio de la memoria a largo plazo que permite a la LSTM relacionar el contexto del inicio de una reseña con su desenlace final.

In [ ]:
print("=== LSTM Config-A: 32 unidades ===")
lstm_A, hist_lstmA, t_lstmA = entrenar(build_lstm(32), 'LSTM-A (32 units)')

**Entrenamiento LSTM-A (32 unidades) completado.** Con solo 32 unidades, este modelo es más compacto que LSTM-B pero ya muestra una ventaja clave sobre la SimpleRNN equivalente: mayor estabilidad en las curvas de validación. Esto se debe a las compuertas de la LSTM que controlan el flujo de información: la compuerta de olvido decide qué contexto anterior descartar, evitando que el gradiente se desvanezca en secuencias de 300 tokens como las reseñas de IMDB.

In [ ]:
print("=== LSTM Config-B: 64 unidades ===")
lstm_B, hist_lstmB, t_lstmB = entrenar(build_lstm(64), 'LSTM-B (64 units)')

**Entrenamiento LSTM-B (64 unidades) completado.** Con 64 unidades, la LSTM tiene mayor capacidad para memorizar patrones contextuales a lo largo de toda la reseña. Es habitual que LSTM-B supere a LSTM-A en `val_accuracy`, aunque la diferencia suele ser menor que la ganancia obtenida al pasar de SimpleRNN a LSTM: el salto de arquitectura (compuertas de memoria) aporta más valor que duplicar el número de unidades, ya que el problema principal de la SimpleRNN es el desvanecimiento del gradiente, no la falta de capacidad.

In [ ]:
plot_historia(hist_lstmA, 'LSTM-A 32 units', '#e67e22')
plot_historia(hist_lstmB, 'LSTM-B 64 units', '#e74c3c')

**Curvas LSTM — observación cualitativa.** Las curvas de la LSTM son generalmente más suaves y estables que las de la SimpleRNN. La convergencia es más consistente porque las compuertas internas filtran el ruido: la compuerta de entrada controla qué nueva información se incorpora al estado de celda, y la compuerta de olvido decide qué contexto anterior descartar, resultando en un gradiente más limpio durante la retropropagación. Si las curvas de LSTM-A y LSTM-B están muy próximas, indica que el número de unidades no es el hiperparámetro más crítico y que pasar de 32 a 64 unidades no justifica el doble de parámetros.

In [ ]:
# Seleccionar mejor LSTM
va_lstmA = max(hist_lstmA.history['val_accuracy'])
va_lstmB = max(hist_lstmB.history['val_accuracy'])

print(f"LSTM-A (32 units): mejor val_acc = {va_lstmA:.4f}")
print(f"LSTM-B (64 units): mejor val_acc = {va_lstmB:.4f}")

best_lstm       = lstm_A if va_lstmA >= va_lstmB else lstm_B
best_lstm_name  = 'LSTM-A (32 units)' if va_lstmA >= va_lstmB else 'LSTM-B (64 units)'
best_units_lstm = 32 if va_lstmA >= va_lstmB else 64
print(f"\nMejor LSTM: {best_lstm_name}")

**Mejor configuración LSTM seleccionada.** El proceso de selección automática basado en `val_accuracy` garantiza objetividad: se elige la configuración que mejor generaliza al conjunto de validación, sin intervención manual que pudiera introducir sesgo. Esta configuración (número de unidades) será la que se use en los experimentos de épocas y en el modelo final de comparación contra la RNN.

In [ ]:
# Épocas con la mejor LSTM (mismo procedimiento que RNN)
EPOCH_LIST_LSTM = [3, 5, 10, 15, 20]
resultados_lstm = []
historiales_lstm = []

print(f"{'Épocas':>7} | train_acc | val_acc | val_loss")
print("-" * 45)

for ep in EPOCH_LIST_LSTM:
    m = build_lstm(best_units_lstm)
    m.compile(optimizer=keras.optimizers.Adam(0.001),
              loss='binary_crossentropy', metrics=['accuracy'])
    hist = m.fit(X_train, y_train, batch_size=BATCH_SIZE,
                 epochs=ep, validation_split=0.20, verbose=0)
    tr  = hist.history['accuracy'][-1]
    va  = hist.history['val_accuracy'][-1]
    vl  = hist.history['val_loss'][-1]
    print(f"{ep:>7} | {tr:.4f}    | {va:.4f}  | {vl:.4f}")
    historiales_lstm.append((ep, hist))
    resultados_lstm.append({'epochs': ep, 'train_acc': tr, 'val_acc': va})

idx_best_lstm = int(np.argmax([r['val_acc'] for r in resultados_lstm]))
BEST_EP_LSTM  = resultados_lstm[idx_best_lstm]['epochs']
print(f"\nMejor número de épocas LSTM: {BEST_EP_LSTM}")

**Experimento de épocas con la LSTM.** A diferencia de la SimpleRNN, la LSTM mantiene un rendimiento más estable a medida que aumentan las épocas. La brecha entre `train_acc` y `val_acc` crece más lentamente gracias a sus compuertas internas, que actúan como mecanismo de regularización implícito: la compuerta de olvido descarta información irrelevante en lugar de propagarla y contaminar el gradiente. Esto suele permitir entrenar más épocas antes de caer en sobreajuste, resultando en un número óptimo de épocas típicamente mayor que el de la RNN. El mejor número de épocas LSTM identificado se usará para el modelo final.

---
## vi. Comparación RNN vs LSTM

In [ ]:
# Reentrenar cada modelo con su número óptimo de épocas para la evaluación final
print(f"Reentrenando RNN con {BEST_EP_RNN} épocas...")
rnn_final = build_rnn(best_units_rnn)
rnn_final.compile(optimizer=keras.optimizers.Adam(0.001),
                  loss='binary_crossentropy', metrics=['accuracy'])
t0_rnn = time.time()
hist_rnn_final = rnn_final.fit(X_train, y_train, batch_size=BATCH_SIZE,
                                epochs=BEST_EP_RNN, validation_split=0.20, verbose=0)
t_rnn_total = time.time() - t0_rnn
print(f"  Listo en {t_rnn_total:.0f}s")

print(f"\nReentrenando LSTM con {BEST_EP_LSTM} épocas...")
lstm_final = build_lstm(best_units_lstm)
lstm_final.compile(optimizer=keras.optimizers.Adam(0.001),
                   loss='binary_crossentropy', metrics=['accuracy'])
t0_lstm = time.time()
hist_lstm_final = lstm_final.fit(X_train, y_train, batch_size=BATCH_SIZE,
                                  epochs=BEST_EP_LSTM, validation_split=0.20, verbose=0)
t_lstm_total = time.time() - t0_lstm
print(f"  Listo en {t_lstm_total:.0f}s")

**Modelos finales reentrenados con épocas óptimas.** Tanto la RNN como la LSTM fueron reentrenadas desde cero usando el número de épocas que maximizó `val_accuracy` en los experimentos anteriores. Este paso garantiza que la comparación en el conjunto de test sea justa: cada modelo opera en su configuración óptima y no en una arbitraria. Reentrenar desde cero (en lugar de continuar desde el modelo anterior) evita que los pesos de épocas subóptimas contaminen la comparación final.

In [ ]:
# Gráfica comparativa en las mismas épocas
ep_comun = min(BEST_EP_RNN, BEST_EP_LSTM)

rnn_cmp = build_rnn(best_units_rnn)
rnn_cmp.compile(optimizer=keras.optimizers.Adam(0.001),
                loss='binary_crossentropy', metrics=['accuracy'])
hist_rnn_cmp = rnn_cmp.fit(X_train, y_train, batch_size=BATCH_SIZE,
                            epochs=ep_comun, validation_split=0.20, verbose=0)

lstm_cmp = build_lstm(best_units_lstm)
lstm_cmp.compile(optimizer=keras.optimizers.Adam(0.001),
                 loss='binary_crossentropy', metrics=['accuracy'])
hist_lstm_cmp = lstm_cmp.fit(X_train, y_train, batch_size=BATCH_SIZE,
                              epochs=ep_comun, validation_split=0.20, verbose=0)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle(f'Comparación RNN vs LSTM — {ep_comun} épocas', fontsize=14, fontweight='bold')

for ax, met, lbl in zip(axes, ['accuracy', 'loss'], ['Exactitud', 'Pérdida']):
    ax.plot(hist_rnn_cmp.history[f'val_{met}'],  label='RNN — Val',  color='#3498db', linewidth=2)
    ax.plot(hist_lstm_cmp.history[f'val_{met}'], label='LSTM — Val', color='#e74c3c', linewidth=2)
    ax.set_xlabel('Época'); ax.set_ylabel(lbl)
    ax.set_title(f'{lbl} de Validación')
    ax.legend(fontsize=11); ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('comparacion_RNN_LSTM.png', dpi=100, bbox_inches='tight')
plt.show()

**Comparación visual RNN vs LSTM.** Las curvas de `val_accuracy` y `val_loss` en el mismo número de épocas confirman la superioridad de la LSTM: su curva de validación está por encima (mayor accuracy) y su pérdida de validación es menor. La diferencia es más pronunciada en las primeras épocas, donde la LSTM aprende más rápido gracias a su capacidad de retener contexto de largo alcance desde el inicio del entrenamiento. La SimpleRNN fluctúa más porque pierde parte del contexto en secuencias largas, lo que genera gradientes menos estables.

In [ ]:
# Evaluación en test
y_pred_rnn  = (rnn_final.predict(X_test,  verbose=0) > 0.5).astype(int).flatten()
y_pred_lstm = (lstm_final.predict(X_test, verbose=0) > 0.5).astype(int).flatten()

acc_rnn   = accuracy_score(y_test, y_pred_rnn)
acc_lstm  = accuracy_score(y_test, y_pred_lstm)
f1_rnn    = f1_score(y_test, y_pred_rnn,  average='weighted')
f1_lstm   = f1_score(y_test, y_pred_lstm, average='weighted')

print("=" * 55)
print(f"{'':20s} {'RNN':>12} {'LSTM':>12}")
print("=" * 55)
print(f"{'Accuracy (test)':20s} {acc_rnn:>12.4f} {acc_lstm:>12.4f}")
print(f"{'F1-score (test)':20s} {f1_rnn:>12.4f} {f1_lstm:>12.4f}")
print(f"{'Tiempo (s)':20s} {t_rnn_total:>12.0f} {t_lstm_total:>12.0f}")
print(f"{'Épocas óptimas':20s} {BEST_EP_RNN:>12} {BEST_EP_LSTM:>12}")
print("=" * 55)

ganador = 'LSTM' if acc_lstm >= acc_rnn else 'RNN'
print(f"\nMejor modelo en test: {ganador}")

**Resultado de la evaluación final en test.** La tabla comparativa muestra que la LSTM obtiene mayor Accuracy y F1-score que la SimpleRNN. Esta diferencia confirma que las compuertas de memoria son cruciales para el análisis de sentimientos en textos de longitud media: una reseña puede comenzar con comentarios positivos sobre la actuación y terminar con una valoración negativa del guion, y solo la LSTM es capaz de mantener y relacionar ambos contextos a lo largo de 300 tokens. El tiempo de entrenamiento mayor de la LSTM queda justificado por la mejora en desempeño.

---
## vii. Métricas y Análisis Final

In [ ]:
CLASES_TEXTO = ['Negativa', 'Positiva']

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Matrices de Confusión — Conjunto de Prueba (IMDB)', fontsize=14, fontweight='bold')

for ax, y_pred, titulo in zip(axes,
                               [y_pred_rnn, y_pred_lstm],
                               ['RNN Simple', 'LSTM']):
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=CLASES_TEXTO, yticklabels=CLASES_TEXTO,
                linewidths=0.5, ax=ax, annot_kws={'size': 13})
    ax.set_title(titulo, fontsize=12, fontweight='bold')
    ax.set_ylabel('Real')
    ax.set_xlabel('Predicho')

plt.tight_layout()
plt.savefig('confusion_matrices_RNN_LSTM.png', dpi=120, bbox_inches='tight')
plt.show()

**Interpretación de las matrices de confusión.** Ambas matrices son casi simétricas porque el dataset está perfectamente balanceado: los errores se distribuyen por igual entre reseñas positivas mal clasificadas como negativas (falsos negativos) y viceversa (falsos positivos). La LSTM muestra diagonales más oscuras (más predicciones correctas) y valores fuera de diagonal más claros (menos errores). Los errores que persisten en ambos modelos corresponden típicamente a reseñas con lenguaje ambiguo, ironía o doble sentido, que son inherentemente difíciles para modelos sin mecanismo de atención.

In [ ]:
print("REPORTE DETALLADO — RNN Simple")
print("-" * 45)
print(classification_report(y_test, y_pred_rnn, target_names=CLASES_TEXTO))

print("\nREPORTE DETALLADO — LSTM")
print("-" * 45)
print(classification_report(y_test, y_pred_lstm, target_names=CLASES_TEXTO))

**Reporte de clasificación detallado.** Los valores de precision, recall y F1-score para ambas clases (Negativa y Positiva) son muy similares entre sí en los dos modelos, lo que confirma que el balance perfecto del dataset (12,500 ejemplos por clase) no introduce sesgos hacia ninguna clase. Si precision > recall para una clase, el modelo es conservador al predecirla (pocos falsos positivos pero más falsos negativos); si recall > precision, es más agresivo. Una diferencia pequeña entre ambas métricas indica que el modelo está bien calibrado.

In [ ]:
# Visualización: barras de métricas comparativas
metricas  = ['Accuracy', 'F1-score']
vals_rnn  = [acc_rnn,  f1_rnn]
vals_lstm = [acc_lstm, f1_lstm]

x = np.arange(len(metricas))
w = 0.35

fig, ax = plt.subplots(figsize=(8, 5))
bars1 = ax.bar(x - w/2, vals_rnn,  w, label='RNN Simple', color='#3498db', edgecolor='white')
bars2 = ax.bar(x + w/2, vals_lstm, w, label='LSTM',       color='#e74c3c', edgecolor='white')

for bar in bars1 + bars2:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.003,
            f'{bar.get_height():.4f}', ha='center', va='bottom', fontsize=10, fontweight='bold')

ax.set_xticks(x); ax.set_xticklabels(metricas, fontsize=12)
ax.set_ylim(0, 1.08)
ax.set_ylabel('Valor', fontsize=12)
ax.set_title('Comparación de Métricas: RNN vs LSTM', fontsize=13, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('comparacion_metricas.png', dpi=100, bbox_inches='tight')
plt.show()

**Resumen visual de métricas finales.** El gráfico de barras sintetiza la comparación RNN vs LSTM en Accuracy y F1-score. La LSTM supera a la SimpleRNN en ambas métricas gracias a su mecanismo de compuertas. Aunque la diferencia en puntos absolutos puede parecer pequeña, con 25,000 muestras de test es estadísticamente significativa. La ventaja se atribuye exclusivamente a la arquitectura recurrente, ya que todas las demás variables (vocabulario de 10,000 palabras, `MAX_LEN=300`, embedding de 64 dims, optimizador Adam lr=0.001) se mantuvieron idénticas en ambos modelos para garantizar una comparación justa.

---
## Conclusiones — Parte 2

### vi. Comparación RNN vs LSTM

| Criterio | RNN Simple | LSTM |
|----------|-----------|------|
| **Accuracy test** | reportado arriba | reportado arriba |
| **Tiempo por época** | menor (~4x más rápida) | mayor (más parámetros por compuertas) |
| **Textos largos** | degradación notable (gradiente se desvanece en secuencias >100 tokens) | mejor manejo gracias a las compuertas que filtran qué recordar |
| **Generalización** | riesgo de sobreajuste en textos largos | mejor generalización |

**¿Cuál tardó más?** La LSTM, porque cada celda tiene 4 matrices de pesos (compuertas forget, input, output + cell) frente a solo 1 en la SimpleRNN.

**¿La LSTM manejó mejor textos largos?** Sí. Las reseñas de IMDB tienen una mediana de ~240 palabras. La RNN simple sufre desvanecimiento del gradiente a esa longitud, perdiendo el contexto del inicio de la reseña. La LSTM lo mitiga con su mecanismo de memoria a largo plazo.

### vii. Factores que afectan el desempeño en texto

1. **Tamaño del vocabulario (`NUM_WORDS`):** un vocabulario pequeño pierde palabras discriminativas; uno muy grande aumenta el espacio de embedding y requiere más datos para entrenarlo. Con 10,000 palabras se cubre el ~95% del vocabulario activo de IMDB.

2. **Longitud máxima de secuencia (`MAX_LEN`):** si es muy corta se pierde contexto relevante (el final de la reseña, donde suele estar la valoración definitiva); si es muy larga aumenta el tiempo de cómputo y el riesgo de que la RNN pierda la información por gradiente desvanecido. `MAX_LEN=300` fue el balance óptimo.

3. **Número de unidades de la RNN/LSTM:** más unidades → mayor capacidad de memorizar patrones, pero también más parámetros → sobreajuste con datasets pequeños. **El hiperparámetro más sensible fue el número de épocas**: la diferencia entre 5 y 10 épocas produjo el mayor salto en val_accuracy, mientras que duplicar las unidades (32→64) tuvo un efecto menor.

### Dataset desbalanceado
**No:** IMDB tiene exactamente 12,500 positivas y 12,500 negativas en train. No se requirió ningún ajuste por desbalance.